In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
from tqdm import tqdm

import kagglehub


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO

dataset_root = os.path.join(path, "dataset")

print("Dataset contents:")
print(os.listdir(dataset_root))

img_dir = os.path.join(dataset_root, "images")
mask_dir = os.path.join(dataset_root, "masks")

print("Images folder sample:")
print(os.listdir(img_dir)[:5])

print("Masks folder sample:")
print(os.listdir(mask_dir)[:5])

image_paths = []
mask_paths = []

# Loop through image folder and match masks by filename
for filename in os.listdir(img_dir):
  if filename.lower().endswith((".jpg", ".jpeg")):
    img_path = os.path.join(img_dir, filename)

    # YOUR CODE HERE
    base = os.path.splitext(filename)[0]
    m_path = os.path.join(mask_dir, base + ".png")

    if os.path.exists(m_path):
      image_paths.append(img_path)
      mask_paths.append(m_path)

print(f"Total images: {len(image_paths)}")
print(f"Total masks: {len(mask_paths)}")


IMG_SIZE = (256, 256)

# Image transforms (Resize, ToTensor, Normalize with ImageNet stats)
image_transforms = transforms.Compose([
  # TODO: Add transforms
  # Hint: ToTensor, Resize to (256, 256), Normalize with ImageNet mean/std
  # YOUR CODE HERE
  transforms.ToTensor(),
  transforms.Resize(IMG_SIZE),
  transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

mask_transforms = transforms.Compose([
  transforms.Resize(IMG_SIZE, interpolation=transforms.InterpolationMode.NEAREST),
  transforms.PILToTensor(),
])

class SUIMDataset(Dataset):
  def __init__(self, image_paths, mask_paths, transform=None, target_transform=None):
    self.image_paths = image_paths
    self.mask_paths = mask_paths
    self.transform = transform
    self.target_transform = target_transform

  def __len__(self):
    # TODO: Return the number of samples in the dataset
    # YOUR CODE HERE
    return len(self.image_paths)

  def __getitem__(self, idx):

    # TODO: Load the image and mask at index idx
    # Hint: Use Image.open() and convert image to "RGB", mask to "L" (grayscale)
    # YOUR CODE HERE
    image = Image.open(self.image_paths[idx]).convert("RGB")
    mask = Image.open(self.mask_paths[idx]).convert("L")

    # Apply transforms
    if self.transform:
      image = self.transform(image)

    if self.target_transform:
      mask = self.target_transform(mask)      # [1,H,W]
      mask = mask.squeeze(0).long()           # [H,W]
      mask = remap_mask(mask)                 # keep consecutive ids

    return image, mask

dataset = SUIMDataset(image_paths, mask_paths, transform=image_transforms, target_transform=mask_transforms)

# Split into train and test sets (80% train, 20% test)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

BATCH_SIZE = 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")


def denormalize(img):
  mean = np.array([0.485, 0.456, 0.406])
  std = np.array([0.229, 0.224, 0.225])
  img = img.permute(1, 2, 0).numpy()
  img = img * std + mean
  img = np.clip(img, 0, 1)
  return img

# Display 4 images with their masks side by side
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(4):
    # YOUR CODE HERE
    image, mask = train_dataset[i]

    axes[0, i].imshow(denormalize(image))
    axes[0, i].set_title(f"Image {i+1}")
    axes[0, i].axis("off")

    axes[1, i].imshow(mask.numpy(), vmin=0, vmax=7)
    axes[1, i].set_title(f"Mask {i+1}")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()



In [ ]:
!pip install -q segmentation_models_pytorch

In [ ]:
# TO DO
import segmentation_models_pytorch as smp
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=8,
)
model = model.to(device)



In [ ]:
# TO DO

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for images, masks in tqdm(dataloader, desc="Train", leave=False):
        images = images.to(device)
        masks = masks.to(device).long()

        # TODO: Complete the training step
        # 1. Forward pass
        # 2. Compute loss
        # 3. Zero gradients
        # 4. Backward pass
        # 5. Update weights

        # YOUR CODE HERE

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    return total_loss / len(dataloader.dataset)


@torch.no_grad()
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0

    for images, masks in tqdm(dataloader, desc="Val", leave=False):
        images = images.to(device)
        masks = masks.to(device).long()

      # TODO: Complete the validation step
      # 1. Forward pass
      # 2. Compute loss

      # YOUR CODE HERE

        logits = model(images)
        loss = criterion(logits, masks)

        total_loss += loss.item() * images.size(0)

    return total_loss / len(dataloader.dataset)


In [ ]:
# TO DO

# YOUR CODE HERE
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 10  # Train for 5 epochs

# Run training

train_losses = []
val_losses = []

for epoch in range(num_epochs):
    tr_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    va_loss = validate(model, val_loader, criterion, device)

    train_losses.append(tr_loss)
    val_losses.append(va_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {tr_loss:.4f}, Val Loss = {va_loss:.4f}")

# Plot loss curve
plt.figure(figsize=(10, 4))
plt.plot(range(1, num_epochs + 1), train_losses, marker="o", label="Train Loss")
plt.plot(range(1, num_epochs + 1), val_losses, marker="o", label="Val Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# TO DO
model.eval()

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
indices = random.sample(range(len(val_dataset)), 4)

for i, idx in enumerate(indices):
    image, mask = val_dataset[idx]

    with torch.no_grad():
        logits = model(image.unsqueeze(0).to(device))
        pred = torch.argmax(logits, dim=1).squeeze(0).cpu()  # [H,W]
  # Display results
    axes[i, 0].imshow(denormalize(image).permute(1, 2, 0))
    axes[i, 0].set_title("Image")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(mask.cpu().numpy(), vmin=0, vmax=7)
    axes[i, 1].set_title("Ground Truth")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(pred.numpy(), vmin=0, vmax=7)
    axes[i, 2].set_title("Prediction")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()